<a href="https://colab.research.google.com/github/bsheese/cs377/blob/18_classification_redesign/18_classification/18_1_Classification_Basics/18_1_1_Confusion_Matrix_Basic_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Confusion Matrix & Basic Classification Metrics

**Author:** Brad Sheese

---

## Introduction

In this notebook, we dive deep into evaluating classification models using the confusion matrix and key metrics: Accuracy, Precision, Recall, and F1-Score.

**Why this matters**: With our imbalanced dataset (~22% defaults), accuracy alone is misleading.

### Learning Objectives
By the end of this notebook, you will be able to:
1. Build and interpret a confusion matrix
2. Understand why accuracy can be misleading on imbalanced data
3. Calculate and interpret Precision, Recall, and F1-Score
4. Recognize when each metric is most important

## Load Data and Train Model

We'll reload the data and retrain our Logistic Regression model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Load and prepare data
data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame

y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

print("Data loaded and model trained!")
print(f"Test set size: {len(y_test)}")
print(f"Actual defaults in test set: {y_test.sum()}")

## The Confusion Matrix

The confusion matrix breaks down predictions into four categories:
- **True Positives (TP)**: Correctly predicted defaults
- **True Negatives (TN)**: Correctly predicted non-defaults
- **False Positives (FP)**: Predicted default but actually good (false alarm)
- **False Negatives (FN)**: Predicted good but actually defaulted (missed cases)

In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Extract values
tn, fp, fn, tp = cm.ravel()
print(f"\nTrue Negatives (TN): {tn} - Correctly predicted 'good'")
print(f"False Positives (FP): {fp} - False alarms (predicted bad, actually good)")
print(f"False Negatives (FN): {fn} - Missed cases (predicted good, actually bad)")
print(f"True Positives (TP): {tp} - Correctly predicted defaults")

# Visualize
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Good', 'Predicted: Bad'],
            yticklabels=['Actual: Good', 'Actual: Bad'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Credit Default Prediction')
plt.tight_layout()
plt.show()

## Accuracy: The Deceptive Metric

Accuracy = (TP + TN) / Total

On imbalanced data, a model that always predicts the majority class can achieve high accuracy!

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Our Logistic Regression Model Accuracy: {accuracy*100:.2f}%")

# Compare to naive baseline (always predict majority class)
baseline_accuracy = (y_test == 0).mean()
print(f"Baseline (always predict 'good'): {baseline_accuracy*100:.2f}%")

print(f"\nThe model is only {(accuracy - baseline_accuracy)*100:.2f}% better than naive!")

# Visualize accuracy
plt.figure(figsize=(8, 5))
metrics = ['Model Accuracy', 'Baseline (Naive)']
values = [accuracy * 100, baseline_accuracy * 100]
bars = plt.bar(metrics, values, color=['steelblue', 'gray'])
plt.ylabel('Accuracy (%)')
plt.title('Accuracy Comparison: Model vs. Naive Baseline')
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', 
             ha='center', fontweight='bold')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

## Precision: How precise are our positive predictions?

Precision = TP / (TP + FP)

Out of all the times we predicted 'default', how often were we right?

**High precision = fewer false alarms**

In [ ]:
# Calculate Precision
precision = precision_score(y_test, y_pred)
print(f"Precision: {precision*100:.2f}%")
print(f"\nInterpretation:")
print(f"  Out of {tp + fp} predictions of 'default', only {tp} were actually defaults.")
print(f"  That's {(precision)*100:.1f}% precision - we have {(fp)} false alarms!")

# What if we raised threshold to be super conservative?
y_pred_conservative = (y_proba >= 0.7).astype(int)  # Higher threshold = more precision
precision_high = precision_score(y_test, y_pred_conservative)
print(f"\nWith threshold=0.7 (conservative), precision improves to: {precision_high*100:.1f}%")
print(f"BUT we catch only {y_pred_conservative.sum()} defaults instead of {y_pred.sum()}")

## Recall: How many of the actual positives did we catch?

Recall = TP / (TP + FN) = 1 - Miss Rate

Out of all the actual defaults, how many did we successfully identify?

**High recall = fewer missed cases**

In [ ]:
# Calculate Recall (Sensitivity / True Positive Rate)
recall = recall_score(y_test, y_pred)
print(f"Recall: {recall*100:.2f}%")
print(f"\nInterpretation:")
print(f"  Out of {tp + fn} actual defaults, we correctly identified only {tp}.")
print(f"  We MISSED {fn} default cases - potentially costly in real world!")

# Lower threshold to catch more defaults
y_pred_sensitive = (y_proba >= 0.3).astype(int)  # Lower threshold = more recall
recall_sensitive = recall_score(y_test, y_pred_sensitive)
print(f"\nWith threshold=0.3 (sensitive), recall improves to: {recall_sensitive*100:.1f}%")
print(f"BUT we also have {(y_pred_sensitive == 1) & (y_test == 0)}.sum() more false alarms!")

## F1-Score: Balancing Precision and Recall

F1 = 2 * (Precision * Recall) / (Precision + Recall)

The Harmonic Mean of precision and recall - penalizes models that sacrifice one for the other.
A score of 1.0 is perfect; 0.0 is worst.

In [ ]:
# Calculate F1-Score
f1 = f1_score(y_test, y_pred)
print(f"F1-Score: {f1*100:.2f}%")
print(f"\nOur metrics at default threshold (0.5):")
print(f"  Precision: {precision*100:.1f}%")
print(f"  Recall:    {recall*100:.1f}%")
print(f"  F1-Score:  {f1*100:.1f}%")

# Visualize all metrics
plt.figure(figsize=(10, 5))
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_values = [accuracy*100, precision*100, recall*100, f1*100]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = plt.bar(metrics_names, metrics_values, color=colors)
plt.ylabel('Score (%)')
plt.title('Classification Metrics at Threshold=0.5')
for bar, val in zip(bars, metrics_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}%', 
             ha='center', fontweight='bold')
plt.ylim(0, 90)
plt.tight_layout()
plt.show()

## When to Use Which Metric?

| Scenario | Metric to Maximize | Reason |
|----------|-------------------|--------|
| Spam detection | **Precision** | False positives annoy users; false negatives are okay |
| Medical diagnosis | **Recall** | Missing a disease is fatal; some false alarms are okay |
| Fraud detection | **Balance** (F1) | Neither false alarms nor missed fraud are acceptable |
| Balanced dataset | **Accuracy** | Works well when classes are ~50/50 |

## Key Takeaways

1. **Confusion Matrix** shows exact breakdown of correct/incorrect predictions
2. **Accuracy fails on imbalanced data** - always check class balance
3. **Precision** - minimize false alarms
4. **Recall** - minimize missed cases
5. **F1-Score** balances both - use when classes are imbalanced